# Wanderbricks Payments Analysis

**Dataset:** `samples.wanderbricks.payments`

**Difficulty:** Medium

**Topics:** aggregation, window, filter on status, date

In [0]:
from pyspark.sql import functions as F, types as T
from pyspark.sql import Window as W

payments = spark.read.table("samples.wanderbricks.payments")

## Problem 1

Total revenue per payment method for completed payments (`status = 'completed'`).
Sort by `total_amount` descending.

**Expected output columns:**
- `payment_method`
- `total_amount`
- `payment_count`

In [0]:
# Problem 1 - write your solution here
# Assign your result to: result_1

result_1 = (
    payments
    .filter(F.col("status") == 'completed')
    .groupBy("payment_method")
    .agg(
        F.sum("amount").alias("total_amount"),
        F.count("payment_id").alias("payment_count")
    )
    .orderBy(F.col("total_amount").desc())
)

result_1.display()

In [0]:
# ── Tests for Problem 1 ──────────────────────────────────────────
assert result_1 is not None, "result_1 is None - did you assign your DataFrame?"
assert hasattr(result_1, 'columns'), "result_1 must be a Spark DataFrame"
cols = [c.lower() for c in result_1.columns]
assert 'payment_method' in cols, "Missing column: payment_method"
assert 'total_amount' in cols, "Missing column: total_amount"
assert 'payment_count' in cols, "Missing column: payment_count"
assert len(cols) == 3, f"Expected exactly 3 columns, got {len(cols)}: {cols}"
cnt = result_1.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
min_amount = result_1.agg(F.min('total_amount')).collect()[0][0]
assert float(min_amount) >= 0, f"Expected total_amount >= 0, found min={min_amount}"
print(f"Problem 1 passed ✓  ({cnt} rows)")

## Problem 2

Calculate the failed payment rate per payment method (count of failed / count of total).

**Expected output columns:**
- `payment_method`
- `total_payments`
- `failed_payments`
- `failure_rate_pct`

In [0]:
# Problem 2 - write your solution here
# Assign your result to: result_2

result_2 = (
    payments
    .groupBy("payment_method")
    .pivot("status")
    .count()
    .withColumn(
        "total_payments",
        (
            F.col("completed")
            + F.col("refunded")
            + F.col("failed")
        ).alias("total_payments")
    )
    .select(
        "payment_method",
        "total_payments",
        F.col("failed").alias("failed_payments"),
        F.round(
            100 * F.col("failed")
            / F.col("total_payments")
        , 2).alias("failure_rate_pct")
    )
)

In [0]:
result_2 = (
    payments
    .groupBy("payment_method")
    .agg(
        F.count("payment_id").alias("total_payments"),
        F.sum(
            F.when(F.col("status") == "failed", 1)
            .otherwise(0)
        ).alias("failed_payments")
    )
    .withColumn(
        "failure_rate_pct",
        F.round(
            100 * F.col("failed_payments")
            / F.col("total_payments")
        , 2)
    )
)

result_2.show()

In [0]:
# ── Tests for Problem 2 ──────────────────────────────────────────
assert result_2 is not None, "result_2 is None - did you assign your DataFrame?"
assert hasattr(result_2, 'columns'), "result_2 must be a Spark DataFrame"
cols = [c.lower() for c in result_2.columns]
assert 'payment_method' in cols, "Missing column: payment_method"
assert 'total_payments' in cols, "Missing column: total_payments"
assert 'failed_payments' in cols, "Missing column: failed_payments"
assert 'failure_rate_pct' in cols, "Missing column: failure_rate_pct"
assert len(cols) == 4, f"Expected exactly 4 columns, got {len(cols)}: {cols}"
cnt = result_2.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
max_rate = result_2.agg(F.max('failure_rate_pct')).collect()[0][0]
min_rate = result_2.agg(F.min('failure_rate_pct')).collect()[0][0]
assert float(max_rate) <= 100, f"Expected failure_rate_pct <= 100, got max={max_rate}"
assert float(min_rate) >= 0, f"Expected failure_rate_pct >= 0, got min={min_rate}"
print(f"Problem 2 passed ✓  ({cnt} rows)")

## Problem 3

Monthly payment volume: count and total amount, extracting year and month from `payment_date`.

**Expected output columns:**
- `year`
- `month`
- `payment_count`
- `total_amount`

In [0]:
# Problem 3 - write your solution here
# Assign your result to: result_3

result_3 = (
    payments
    .groupBy(F.year("payment_date").alias("year"), F.month("payment_date").alias("month"))
    .agg(F.count("payment_id").alias("payment_count"), F.sum("amount").alias("total_amount"))
    .orderBy("year", "month")
)

result_3.show()

In [0]:
# ── Tests for Problem 3 ──────────────────────────────────────────
assert result_3 is not None, "result_3 is None - did you assign your DataFrame?"
assert hasattr(result_3, 'columns'), "result_3 must be a Spark DataFrame"
cols = [c.lower() for c in result_3.columns]
assert 'year' in cols, "Missing column: year"
assert 'month' in cols, "Missing column: month"
assert 'payment_count' in cols, "Missing column: payment_count"
assert 'total_amount' in cols, "Missing column: total_amount"
assert len(cols) == 4, f"Expected exactly 4 columns, got {len(cols)}: {cols}"
cnt = result_3.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
max_month = result_3.agg(F.max('month')).collect()[0][0]
assert max_month <= 12, f"Expected month <= 12, got max={max_month}"
print(f"Problem 3 passed ✓  ({cnt} rows)")

## Problem 4

Using a window function, compute the running total payment amount ordered by `payment_date`.

**Expected output columns:**
- `payment_id`
- `payment_date`
- `amount`
- `running_total`

In [0]:
# Problem 4 - write your solution here
# Assign your result to: result_4
w = W.orderBy("payment_date").rowsBetween(W.unboundedPreceding, W.currentRow)
result_4 = (
    payments
    .select(
        "payment_id",
        "payment_date",
        "amount",
        F.sum("amount").over(w).alias("running_total")
    )
    .orderBy("payment_date")
)

result_4.limit(5).show()


In [0]:
# ── Tests for Problem 4 ──────────────────────────────────────────
assert result_4 is not None, "result_4 is None - did you assign your DataFrame?"
assert hasattr(result_4, 'columns'), "result_4 must be a Spark DataFrame"
cols = [c.lower() for c in result_4.columns]
assert 'payment_id' in cols, "Missing column: payment_id"
assert 'payment_date' in cols, "Missing column: payment_date"
assert 'amount' in cols, "Missing column: amount"
assert 'running_total' in cols, "Missing column: running_total"
assert len(cols) == 4, f"Expected exactly 4 columns, got {len(cols)}: {cols}"
cnt = result_4.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
max_running = result_4.agg(F.max('running_total')).collect()[0][0]
assert float(max_running) > 0, f"Expected running_total > 0, got max={max_running}"
print(f"Problem 4 passed ✓  ({cnt} rows)")

## Problem 5

Find bookings that have more than one payment record.

**Expected output columns:**
- `booking_id`
- `payment_count`
- `total_paid`

In [0]:
# Problem 5 - write your solution here
# Assign your result to: result_5

result_5 = (
    payments
    .groupBy("booking_id")
    .agg(
        F.count("payment_id").alias("payment_count"),
        F.sum("amount").alias("total_paid")
    )
    .filter(F.col("payment_count") > 1)
)

result_5.limit(5).show()

In [0]:
# ── Tests for Problem 5 ──────────────────────────────────────────
assert result_5 is not None, "result_5 is None - did you assign your DataFrame?"
assert hasattr(result_5, 'columns'), "result_5 must be a Spark DataFrame"
cols = [c.lower() for c in result_5.columns]
assert 'booking_id' in cols, "Missing column: booking_id"
assert 'payment_count' in cols, "Missing column: payment_count"
assert 'total_paid' in cols, "Missing column: total_paid"
assert len(cols) == 3, f"Expected exactly 3 columns, got {len(cols)}: {cols}"
cnt = result_5.count()
assert cnt >= 0, f"Expected rows >= 0, got {cnt}"
if cnt > 0:
    min_pc = result_5.agg(F.min('payment_count')).collect()[0][0]
    assert min_pc > 1, f"Expected payment_count > 1, found min={min_pc}"
print(f"Problem 5 passed ✓  ({cnt} rows)")

## Problem 6

Calculate the percentage of revenue from each payment method out of total completed revenue.

**Expected output columns:**
- `payment_method`
- `total_amount`
- `revenue_pct`

In [0]:
# Problem 6 - write your solution here
# Assign your result to: result_6
w = W.rowsBetween(W.unboundedPreceding, W.unboundedFollowing)
result_6 = (
    payments
    .groupBy("payment_method")
    .agg(
        F.sum("amount").alias("total_amount")
    )
    .withColumn(
        "revenue_pct",
        F.round(
            100 * F.col("total_amount")
            / F.sum("total_amount").over(w)
        , 2)
    )
)

result_6.show()

In [0]:
# ── Tests for Problem 6 ──────────────────────────────────────────
assert result_6 is not None, "result_6 is None - did you assign your DataFrame?"
assert hasattr(result_6, 'columns'), "result_6 must be a Spark DataFrame"
cols = [c.lower() for c in result_6.columns]
assert 'payment_method' in cols, "Missing column: payment_method"
assert 'total_amount' in cols, "Missing column: total_amount"
assert 'revenue_pct' in cols, "Missing column: revenue_pct"
assert len(cols) == 3, f"Expected exactly 3 columns, got {len(cols)}: {cols}"
cnt = result_6.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
max_pct = result_6.agg(F.max('revenue_pct')).collect()[0][0]
min_pct = result_6.agg(F.min('revenue_pct')).collect()[0][0]
assert float(max_pct) <= 100, f"Expected revenue_pct <= 100, got max={max_pct}"
assert float(min_pct) >= 0, f"Expected revenue_pct >= 0, got min={min_pct}"
print(f"Problem 6 passed ✓  ({cnt} rows)")

## Problem 7

Find the average time between payment dates for bookings with multiple payments.
Use a lag window function to get the previous payment date, then compute days between.

**Expected output columns:**
- `booking_id`
- `payment_id`
- `payment_date`
- `prev_payment_date`
- `days_between`

In [0]:
# Problem 7 - write your solution here
# Assign your result to: result_7
w = W.partitionBy("booking_id").orderBy("payment_date")
result_7 = (
    payments
    .select("booking_id", "payment_id", "payment_date")
    .withColumn("prev_payment_date", F.lag("payment_date").over(w))
    .filter(F.col("prev_payment_date").isNotNull())
    .withColumn("days_between", F.datediff("payment_date", "prev_payment_date"))
    .orderBy(F.col("days_between").desc())
)

result_7.limit(5).show()

In [0]:
# ── Tests for Problem 7 ──────────────────────────────────────────
assert result_7 is not None, "result_7 is None - did you assign your DataFrame?"
assert hasattr(result_7, 'columns'), "result_7 must be a Spark DataFrame"
cols = [c.lower() for c in result_7.columns]
assert 'booking_id' in cols, "Missing column: booking_id"
assert 'payment_id' in cols, "Missing column: payment_id"
assert 'payment_date' in cols, "Missing column: payment_date"
assert 'prev_payment_date' in cols, "Missing column: prev_payment_date"
assert 'days_between' in cols, "Missing column: days_between"
assert len(cols) == 5, f"Expected exactly 5 columns, got {len(cols)}: {cols}"
cnt = result_7.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
print(f"Problem 7 passed ✓  ({cnt} rows)")